# Criminalidad en Chicago: análisis bayesiano

Versión para repositorio basada en `notebook_6.ipynb`. Lea primero
`docs/REPRODUCIBILITY.md` y `docs/NOTAS_DE_VERSION.md`.
El póster y el informe se conservan como documentos originales; contienen cifras
que no coinciden completamente con las salidas archivadas. Se retiró la antigua
sección 7.5 con datos simulados. No se ha vuelto a ejecutar el muestreo en esta preparación.


In [ ]:
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "stan_models/poisson_offset.stan").exists()
             and (p / "data/processed/final_data_2.0.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Abra el notebook desde el repositorio descargado.")
RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"
RESULTS_DIR = ROOT / "results/generated"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# 📊 Data Dictionary – Crime & Socioeconomic Dataset (Chicago Community Areas)

| **Columna** | **Tipo de dato** | **Descripción** | **Ejemplo** |
|--------------|------------------|------------------|--------------|
| **Community Area Number** | `int64` | Identificador numérico único del community area dentro de la ciudad. | `25` |
| **COMMUNITY AREA NAME** | `object` | Nombre de la community area. | `Austin` |
| **Total Population** | `int64` | Población total estimada del área. | `98514` |
| **total_crimes** | `int64` | Número total de delitos registrados en el área durante el periodo de estudio. | `12543` |
| **crime_rate** | `float64` | Tasa de criminalidad (delitos por habitante). | `0.127` |
| **total_arrests** | `int64` | Número total de arrestos realizados en el área. | `3421` |
| **arrest_rate_total** | `float64` | Proporción de delitos que resultaron en arresto (`total_arrests / total_crimes`). | `0.27` |
| **violent** | `int64` | Número total de delitos violentos (ej. homicidio, asalto, agresión). | `2310` |
| **pct_violent** | `float64` | Proporción de delitos violentos respecto al total (`violent / total_crimes`). | `0.18` |
| **violent_rate** | `float64` | Tasa de delitos violentos por habitante. | `0.023` |
| **violent_arrests** | `int64` | Número de arrestos asociados a delitos violentos. | `890` |
| **arrest_rate_violent** | `float64` | Proporción de delitos violentos que resultaron en arresto (`violent_arrests / violent`). | `0.38` |
| **property** | `int64` | Número total de delitos contra la propiedad (hurto, robo, etc.). | `7200` |
| **pct_property** | `float64` | Proporción de delitos contra la propiedad respecto al total. | `0.57` |
| **property_rate** | `float64` | Tasa de delitos contra la propiedad por habitante. | `0.073` |
| **property_arrests** | `int64` | Número de arrestos asociados a delitos contra la propiedad. | `1800` |
| **arrest_rate_property** | `float64` | Proporción de delitos contra la propiedad que resultaron en arresto. | `0.25` |
| **other** | `int64` | Número de delitos clasificados como “otros” (no violentos ni de propiedad). | `3033` |
| **pct_other** | `float64` | Proporción de delitos “otros” respecto al total. | `0.24` |
| **other_rate** | `float64` | Tasa de delitos “otros” por habitante. | `0.031` |
| **other_arrests** | `int64` | Número de arrestos asociados a delitos “otros”. | `731` |
| **arrest_rate_other** | `float64` | Proporción de delitos “otros” que resultaron en arresto. | `0.24` |
| **PERCENT OF HOUSING CROWDED** | `float64` | Porcentaje de viviendas con hacinamiento (más de un ocupante por habitación). | `7.5` |
| **PERCENT HOUSEHOLDS BELOW POVERTY** | `float64` | Porcentaje de hogares por debajo de la línea de pobreza. | `22.3` |
| **PERCENT AGED 16+ UNEMPLOYED** | `float64` | Porcentaje de la población mayor de 16 años desempleada. | `12.1` |
| **PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA** | `float64` | Porcentaje de adultos sin diploma de secundaria. | `18.7` |
| **PERCENT AGED UNDER 18 OR OVER 64** | `float64` | Porcentaje de población dependiente (menores de 18 o mayores de 64). | `35.2` |
| **PER CAPITA INCOME** | `int64` | Ingreso per cápita promedio del área (en USD). | `21500` |
| **HARDSHIP INDEX** | `float64` | Índice compuesto de condiciones socioeconómicas adversas (mayor = peor situación). | `72.5` |

---

### 📌 Notas
- Las proporciones (`pct_*`, `arrest_rate_*`) están en escala `[0,1]`.  
- Las variables de criminalidad están desagregadas en tres categorías: **violent**, **property** y **other**.  
- El **HARDSHIP INDEX** es un indicador sintético que combina múltiples dimensiones socioeconómicas.  

# ***Fase 0* - Importaciones**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
plt.style.use("seaborn-v0_8")
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv(PROCESSED_DIR / "final_data_2.0.csv")
display(df.head())


In [ ]:
print(df.describe().T.to_string())

# **Fase 1: Preparación de Datos y EDA:**

## 1 - Matriz de correlación inicial:

De las 29 variables del dataset, no todas son interesantes en cuanto a conocer su correlación...

Elegiremos a continuación las que sí resultan relevantes para nuestros análisis.

In [ ]:
cols_corr = [
    # Crime rates
    "crime_rate",
    "violent_rate",
    "property_rate",
    "other_rate",
    # Arrest efficiency
    "arrest_rate_total",
    "arrest_rate_violent",
    "arrest_rate_property",
    "arrest_rate_other",
    # Socioeconomic
    "PERCENT OF HOUSING CROWDED",
    "PERCENT HOUSEHOLDS BELOW POVERTY",
    "PERCENT AGED 16+ UNEMPLOYED",
    "PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA",
    "PERCENT AGED UNDER 18 OR OVER 64",
    "PER CAPITA INCOME",
    #"HARDSHIP INDEX"  ##recordar que este índice socioeconómico es una combinación de los anteriores, por lo que podría introducir redundancia en la matriz de correlación.
]
len(cols_corr)

In [ ]:
import matplotlib.patches as patches

# Calcular la matriz de correlación
corr_matrix = df[cols_corr].corr()

# Crear máscara para ocultar el triángulo superior (sin incluir diagonal)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Configurar figura
plt.figure(figsize=(16, 16))
ax = plt.gca()

# Dibujar el heatmap (solo triángulo inferior)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt=".4f",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": .8},
    ax=ax
)
ax.grid(False)  # prevenir que aparezcan líneas de cuadrícula sobre el heatmap

# Colorear la diagonal con un tono que la distinga del resto de la matriz
cmap_diag = plt.cm.Greys
diag_color = cmap_diag(0.85)

for i in range(len(corr_matrix)):
    square = patches.Rectangle(
        (i, i), 1, 1,
        linewidth=0,
        edgecolor='none',
        facecolor=diag_color,
        alpha=0.4
    )
    ax.add_patch(square)

# Rotar etiquetas y mejorar legibilidad
plt.xticks(rotation=40, ha='right')
# plt.yticks(rotation=45)
plt.title("Matriz de correlación — Chicago Crime Data", pad=20, c='green')
plt.tight_layout()
# guardar la imagen:
#plt.savefig("matriz_correlacion__Chicago_crime.png", dpi=900, bbox_inches='tight')
plt.show()  # mostrarla

## 2 - Matriz de correlación *minimalista* y cálculo del *VIF* (Factor de Inflación de la Varianza) :

Ahora, vamos a construir una matriz de correlación más minimalista: Consideraremos únicamente las variables socioeconómicas predictoras y el conteo total de crímenes por área...

In [ ]:
selected = [
    "total_crimes",
    "Total Population",
    "PERCENT HOUSEHOLDS BELOW POVERTY",
    "PERCENT AGED 16+ UNEMPLOYED",
    "PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA",
    "PERCENT OF HOUSING CROWDED",
    "PER CAPITA INCOME",
    "HARDSHIP INDEX"
]
# display(df[selected].describe().T)
# corr = df[selected].corr()
# display(corr)
# plt.figure(figsize=(9, 7))
# sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
# plt.title("Correlación entre variables clave")
# plt.show()


import matplotlib.patches as patches

# Calcular la matriz de correlación
corr_matrix = df[selected].corr()

# Crear máscara para ocultar el triángulo superior (sin incluir diagonal)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Configurar figura
plt.figure(figsize=(12, 12))
ax = plt.gca()

# Dibujar el heatmap (solo triángulo inferior)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt=".4f",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": .8},
    ax=ax
)
ax.grid(False)  # prevenir que aparezcan líneas de cuadrícula sobre el heatmap

# Colorear la diagonal con un tono que la distinga del resto de la matriz
cmap_diag = plt.cm.Greys
diag_color = cmap_diag(0.85)

for i in range(len(corr_matrix)):
    square = patches.Rectangle(
        (i, i), 1, 1,
        linewidth=0,
        edgecolor='none',
        facecolor=diag_color,
        alpha=0.4
    )
    ax.add_patch(square)

# Rotar etiquetas y mejorar legibilidad
plt.xticks(rotation=40, ha='right')
# plt.yticks(rotation=45)
plt.title("Matriz de correlación minimalista — Chicago Crime Data", pad=20, c='green')
plt.tight_layout()
# guardar la imagen:
#plt.savefig("matriz_correlacion_minimalista__Chicago_crime.png", dpi=900, bbox_inches='tight')
plt.show()  # mostrarla

En este punto, calcularemos el VIF (Factor de Inflación de la Varianza)... La explicación de lo que este índice significa, así como la interpretación de resultados, se encuentra más abajo.

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calcular_vif(df, columnas):
    X = sm.add_constant(df[columnas])

    vif = pd.DataFrame({
        "Variable": X.columns,
        "VIF": [
            round(variance_inflation_factor(X.values, i), 4)
            for i in range(X.shape[1])
        ]
    })

    return vif[vif["Variable"] != "const"].reset_index(drop=True)

In [ ]:
X = df[selected].drop(columns=["total_crimes"])
X

In [ ]:
calcular_vif(X, X.columns)

### Concepto e interpretación del *Factor de Inflación de la Varianza (VIF)*

Antes de ajustar un modelo de regresión es importante verificar si las variables explicativas presentan **multicolinealidad**, es decir, si algunas de ellas contienen información muy similar. La multicolinealidad no impide estimar un modelo, pero sí puede dificultar la interpretación de los coeficientes, aumentar su varianza y hacer que pequeñas modificaciones en los datos produzcan cambios importantes en las estimaciones.

Una de las herramientas más utilizadas para diagnosticar este fenómeno es el **Factor de Inflación de la Varianza** (*Variance Inflation Factor*, **VIF**).

Para cada variable explicativa $X_j$, el VIF se calcula ajustando una regresión auxiliar en la que dicha variable se explica mediante todas las demás variables del modelo,

$$
X_j=\alpha+\sum_{k\neq j}\beta_kX_k+\varepsilon,
$$

donde el coeficiente de determinación de esta regresión se denota por $R_j^2$. El VIF se define como

$$
\mathrm{VIF}_j=\frac{1}{1-R_j^2}.
$$

Esta expresión cuantifica cuánto aumenta la varianza del estimador del coeficiente asociado a $X_j$ debido a la correlación lineal existente entre esa variable y el resto de los predictores. Cuanto mayor sea el VIF, mayor será la redundancia de información entre las variables explicativas.

La interpretación habitual del VIF es la siguiente:

| VIF | Interpretación |
|------|----------------|
| $1$ | No existe multicolinealidad. |
| $1 < \mathrm{VIF} < 5$ | Multicolinealidad baja o moderada. |
| $5 \leq \mathrm{VIF} < 10$ | Multicolinealidad considerable; conviene analizarla con mayor detalle. |
| $\mathrm{VIF} \geq 10$ | Multicolinealidad severa. |

Es importante destacar que el VIF **no mide la relación entre una variable explicativa y la variable respuesta**, sino únicamente la dependencia lineal existente entre los propios predictores.

En este estudio se obtuvieron los siguientes valores aproximados de VIF:

| Variable | VIF |
|-----------|----:|
| Total Population | 1.48 |
| Percent Households Below Poverty | 5.53 |
| Percent Aged 16+ Unemployed | 7.79 |
| Percent Aged 25+ Without High School Diploma | 11.44 |
| Percent of Housing Crowded | 5.87 |
| Per Capita Income | 4.62 |
| Hardship Index | 35.92 |

Los resultados muestran que **Total Population** presenta un VIF muy cercano a 1, lo que indica que prácticamente no comparte información lineal con las demás variables. Por su parte, **Per Capita Income** presenta un nivel moderado de multicolinealidad, mientras que las variables relacionadas con pobreza, desempleo, nivel educativo y hacinamiento presentan valores superiores a 5, indicando una dependencia importante con el resto de los indicadores socioeconómicos.

El caso más llamativo corresponde al **Hardship Index**, cuyo VIF es aproximadamente **35.9**. Este valor implica que la regresión auxiliar utilizada para calcular el VIF posee un coeficiente de determinación cercano a

$$
R^2 = 1-\frac{1}{35.92}\approx0.972
$$

lo que significa que aproximadamente el **97 % de la variabilidad del Hardship Index puede explicarse mediante las demás variables explicativas del modelo**. En otras palabras, el índice contiene información que ya está presente, casi por completo, en el resto de los predictores.

Este resultado es consistente con la naturaleza del **Hardship Index**, ya que se trata de un **índice compuesto** construido precisamente a partir de diversos indicadores socioeconómicos como pobreza, desempleo, nivel educativo, hacinamiento e ingreso. En consecuencia, al incluir simultáneamente el índice y las variables que lo componen se introduce una redundancia considerable en el modelo, lo que se refleja en un VIF extremadamente elevado.

Es importante señalar que este comportamiento **no necesariamente puede detectarse únicamente mediante la matriz de correlaciones**. Mientras que la matriz de correlaciones analiza la relación entre pares de variables, el VIF evalúa la capacidad de **todas las variables restantes en conjunto** para explicar una variable determinada. Por ello, una variable puede no presentar correlaciones individuales extremadamente altas con ninguna otra y, aun así, tener un VIF muy elevado cuando la combinación lineal del resto de variables logra reproducirla con gran precisión.

Desde el punto de vista del modelamiento estadístico, estos resultados sugieren que no es recomendable incluir simultáneamente el **Hardship Index** y los indicadores socioeconómicos que lo conforman. Una estrategia más apropiada consiste en optar por una de las siguientes alternativas:

1. Utilizar únicamente el **Hardship Index** como un indicador global de las condiciones socioeconómicas.
2. Excluir el **Hardship Index** y conservar las variables individuales (pobreza, desempleo, educación, hacinamiento e ingreso), permitiendo analizar el efecto específico de cada una de ellas sobre la variable respuesta.

En este trabajo resulta más conveniente la segunda alternativa, ya que el objetivo es estudiar la influencia particular de cada factor socioeconómico sobre la criminalidad. Eliminar el **Hardship Index** reduce la redundancia entre los predictores y facilita una interpretación más clara de los coeficientes estimados.

## 3 - Exploración inicial de la variable respuesta y del offset:

La variable objetivo es `total_crimes`. El offset será `log(Total Population)` para modelar la tasa esperada de crímenes por persona en cada área.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Histograma
sns.histplot(df["total_crimes"], bins=15, kde=False,
             color="tab:blue", ax=axes[0])
axes[0].set_title("Distribución de total_crimes")
axes[0].set_xlabel("Total de crímenes")
axes[0].set_ylabel("Número de áreas")
# Dispersión + regresión lineal
sns.regplot(
    x="Total Population",
    y="total_crimes",
    data=df,
    ax=axes[1],
    scatter_kws={"color": "tab:orange"},
    #line_kws={"color": "red"}
)
axes[1].set_title("Total de crímenes vs población")
axes[1].set_xlabel("Población total")
axes[1].set_ylabel("Total de crímenes")
plt.tight_layout()
plt.show()


## 4 - Preparación de las variables para los modelos:

Estandarizamos las covariables socioeconómicas para que los coeficientes sean comparables en magnitud, y creamos el offset logarítmico de población.

In [ ]:
df.columns.to_list()

In [ ]:
predictors = [
 'PERCENT OF HOUSING CROWDED',
 'PERCENT HOUSEHOLDS BELOW POVERTY',
 'PERCENT AGED 16+ UNEMPLOYED',
 'PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA',
 'PERCENT AGED UNDER 18 OR OVER 64',
 'PER CAPITA INCOME',
 'HARDSHIP INDEX']
df_model = df[["Community Area Number", "COMMUNITY AREA NAME", "total_crimes", "Total Population"]
              + predictors].copy()
for col in predictors:
    df_model[col + "_std"] = (df_model[col] - df_model[col].mean()) / df_model[col].std()
predictors_std = [col + "_std" for col in predictors]
df_model["offset"] = np.log(df_model["Total Population"])
print((df_model[predictors_std + ["offset"]]).to_string())
print("\n\n\n")
print(df_model.to_string())


In [ ]:
display(df_model[predictors_std + ["offset"]])
print()
display(df_model)

# **Fase 2: Especificación y Programación de Modelos Probabilísticos:**

In [ ]:
import cmdstanpy
print("CmdStan:", cmdstanpy.cmdstan_path())


In [ ]:
# Instale CmdStan una sola vez siguiendo docs/REPRODUCIBILITY.md.


In [ ]:
from cmdstanpy import CmdStanModel
STAN_DIR = ROOT / "stan_models"

# ==========================================================
# Especificaciones de los modelos
# ==========================================================

model_specs = {
    'M1': {
        'distribution': 'poisson',
        'covariables': [
            'PERCENT HOUSEHOLDS BELOW POVERTY_std',
            'PERCENT AGED 16+ UNEMPLOYED_std'
        ]
    },
    'M2': {
        'distribution': 'poisson',
        'covariables': [
            'PERCENT HOUSEHOLDS BELOW POVERTY_std',
            'PERCENT AGED 16+ UNEMPLOYED_std',
            'PER CAPITA INCOME_std'
        ]
    },
    'M3': {
        'distribution': 'poisson',
        'covariables': [
            'PERCENT HOUSEHOLDS BELOW POVERTY_std',
            'PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA_std',
            'PERCENT OF HOUSING CROWDED_std'
        ]
    },
    'M4': {
        'distribution': 'neg_binomial',
        'covariables': [
            'PERCENT HOUSEHOLDS BELOW POVERTY_std',
            'PERCENT AGED 16+ UNEMPLOYED_std'
        ]
    },
    'M5': {
        'distribution': 'neg_binomial',
        'covariables': [
            'PERCENT HOUSEHOLDS BELOW POVERTY_std',
            'PERCENT AGED 16+ UNEMPLOYED_std',
            'PER CAPITA INCOME_std'
        ]
    }
}

# ==========================================================
# Compilar los modelos Stan
# ==========================================================

stan_models = {
    'poisson': CmdStanModel(
        stan_file=str(STAN_DIR / 'poisson_offset.stan')
    ),
    'neg_binomial': CmdStanModel(
        stan_file=str(STAN_DIR / 'neg_binomial_offset.stan')
    ),
}

# ==========================================================
# Preparar los datos
# ==========================================================

stan_data = {}

for model_name, spec in model_specs.items():
    covs = spec['covariables']

    stan_data[model_name] = {
        'N': len(df_model),
        'K': len(covs),
        'X': df_model[covs].to_numpy(),
        'y': df_model['total_crimes'].astype(int).to_numpy(),
        'log_offset': df_model['offset'].to_numpy(),
    }

# ==========================================================
# Resumen
# ==========================================================

print("Modelos Stan compilados:")
for dist, model in stan_models.items():
    print(f"  - {dist}: {model.stan_file}")

print("\nEspecificaciones cargadas para Fase 2:")
for model_name, spec in model_specs.items():
    print(f"  - {model_name}: {spec['distribution']} con {len(spec['covariables'])} covariables")

print("\nEjemplo de datos para M1:")
print("  N =", stan_data['M1']['N'])
print("  K =", stan_data['M1']['K'])
print("  X shape =", stan_data['M1']['X'].shape)
print("  y shape =", stan_data['M1']['y'].shape)
print("  log_offset shape =", stan_data['M1']['log_offset'].shape)

# **Fase 3: Muestreo Computacional (MCMC vía HMC/NUTS) :**

In [ ]:
import time

# ==========================================================
# Fase 3: Muestreo MCMC mediante HMC/NUTS
# ==========================================================

fits = {}
sampling_times = {}

CHAINS = 4
WARMUP = 1000
SAMPLING = 1000
SEED = 1234

print("=" * 70)
print("INICIANDO AJUSTE DE LOS MODELOS")
print("=" * 70)

for model_name, spec in model_specs.items():

    print(f"\n>>> Ajustando {model_name} ({spec['distribution']})")

    model = stan_models[spec["distribution"]]

    start = time.time()

    fit = model.sample(
        data=stan_data[model_name],
        chains=CHAINS,
        iter_warmup=WARMUP,
        iter_sampling=SAMPLING,
        parallel_chains=CHAINS,
        seed=SEED,
        show_progress=True
    )

    elapsed = time.time() - start

    fits[model_name] = fit
    sampling_times[model_name] = elapsed

    print(f"Tiempo: {elapsed:.2f} segundos")

print("\n" + "=" * 70)
print("MUESTREO FINALIZADO")
print("=" * 70)

print("\nResumen de tiempos")

for model_name, t in sampling_times.items():
    print(f"{model_name}: {t:.2f} s")

# **Fase 4: Diagnóstico de Convergencia y Validación del Ajuste:**

In [ ]:
import pandas as pd

# ==========================================================
# Fase 4.1: Diagnóstico de convergencia
# ==========================================================

diagnostics = []

print("=" * 80)
print("DIAGNÓSTICOS DE CONVERGENCIA")
print("=" * 80)

for model_name, fit in fits.items():

    summary = fit.summary()

    max_rhat = summary["R_hat"].max()
    min_bulk = summary["ESS_bulk"].min()
    min_tail = summary["ESS_tail"].min()

    # Número de divergencias
    sampler_df = fit.draws_pd(vars=["divergent__"])
    divergences = int(sampler_df["divergent__"].sum())

    estado = "✅ OK"

    if max_rhat > 1.01:
        estado = "⚠️ Revisar R̂"

    elif min(min_bulk, min_tail) < 400:
        estado = "⚠️ ESS bajo"

    elif divergences > 0:
        estado = "⚠️ Divergencias"

    diagnostics.append({
        "Modelo": model_name,
        "Rhat_max": round(max_rhat, 4),
        "ESS_bulk_min": int(min_bulk),
        "ESS_tail_min": int(min_tail),
        "Divergencias": divergences,
        "Tiempo (s)": round(sampling_times[model_name], 2),
        "Estado": estado
    })

diagnostics_df = (
    pd.DataFrame(diagnostics)
    .sort_values("Modelo")
    .reset_index(drop=True)
)

display(diagnostics_df)

In [ ]:
print(diagnostics_df.to_string())

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # ==========================================================
# # Fase 4.2: Posterior Predictive Checks
# # ==========================================================

# observed = stan_data["M1"]["y"]

# for model_name, fit in fits.items():

#     yrep = fit.stan_variable("y_rep")

#     # Media posterior de las réplicas
#     yrep_mean = yrep.mean(axis=0)

#     plt.figure(figsize=(7,4))

#     plt.hist(
#         observed,
#         bins=15,
#         alpha=0.6,
#         density=True,
#         label="Observados"
#     )

#     plt.hist(
#         yrep_mean,
#         bins=15,
#         alpha=0.6,
#         density=True,
#         label="Posterior Predictive"
#     )

#     plt.title(f"PPC - {model_name}")

#     plt.xlabel("Conteo de crímenes")

#     plt.ylabel("Densidad")

#     plt.legend()

#     plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Fase 4.2: Posterior Predictive Checks (versión mejorada)
# ==========================================================

rng = np.random.default_rng(123)

observed = stan_data["M1"]["y"]

for model_name, fit in fits.items():
    yrep = fit.stan_variable("y_rep")
    plt.figure(figsize=(8,5))

    # Histograma observado
    plt.hist(
        observed,
        bins=15,
        density=True,
        alpha=0.7,
        label="Observados"
    )

    # Elegimos 100 datasets simulados al azar
    draws = rng.choice(yrep.shape[0], size=100, replace=False)

    for i in draws:

        plt.hist(
            yrep[i],
            bins=15,
            density=True,
            histtype="step",
            alpha=0.05
        )

    plt.title(f"PPC - {model_name}")
    plt.xlabel("Conteo de crímenes")
    plt.ylabel("Densidad")
    plt.legend()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# PPC: Observado vs Predicción con error relativo
# ==========================================================

observed = stan_data["M1"]["y"]
models_summary = []
for model_name, fit in fits.items():

    yrep = fit.stan_variable("y_rep")

    pred_mean = yrep.mean(axis=0)

    lower = np.percentile(yrep, 2.5, axis=0)
    upper = np.percentile(yrep, 97.5, axis=0)

    # Error relativo (%)
    rel_error = np.abs(pred_mean - observed) / np.maximum(observed, 1)

    # Clasificación por colores
    colors = np.where(
        rel_error < 0.10,
        "green",
        np.where(rel_error < 0.25, "orange", "red")
    )

    plt.figure(figsize=(8, 8),
               #dpi=900
               )

    # Dibujar cada punto con su color
    for i in range(len(observed)):
        plt.errorbar(
            observed[i],
            pred_mean[i],
            yerr=[[pred_mean[i] - lower[i]], [upper[i] - pred_mean[i]]],
            fmt='o',
            color=colors[i],
            alpha=0.75,
            capsize=2
        )

    # Recta identidad
    lim = max(observed.max(), upper.max()) * 1.05

    plt.plot(
        [0, lim],
        [0, lim],
        'k--',
        linewidth=2,
        label="Recta identidad"
    )

    # Etiquetar las 5 observaciones con mayor error
    idx = np.argsort(rel_error)[-5:]

    legend_text = "Áreas destacadas\n\n"

    for i in idx:
        area_number = int(df_model.iloc[i]["Community Area Number"])
        area_name = df_model.iloc[i]["COMMUNITY AREA NAME"]

        # Número sobre el punto
        plt.text(
            observed[i],
            pred_mean[i],
            str(area_number),
            fontsize=9,
            weight="bold",
            ha="left",
            va="bottom"
        )

        # Texto para la caja
        legend_text += f"{area_number:>2} : {area_name}\n"

    # Leyenda manual
    from matplotlib.lines import Line2D

    legend_elements = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor='green', markersize=8,
               label='Error < 10 %'),
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor='orange', markersize=8,
               label='10 % ≤ Error < 25 %'),
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor='red', markersize=8,
               label='Error ≥ 25 %'),
        Line2D([0], [0], color='k', linestyle='--',
               label='Recta identidad')
    ]

    plt.legend(handles=legend_elements, loc="upper left")
    plt.grid(True, alpha=0.3)

    # Caja con nombres de las áreas
    plt.gcf().text(
        0.73,               # posición horizontal
        0.25,               # posición vertical
        legend_text,
        fontsize=9,
        family="monospace",
        bbox=dict(
            facecolor="white",
            edgecolor="gray",
            alpha=0.9,
            boxstyle="round,pad=0.5"
        )
    )
    plt.title(f"PPC - {model_name}: Observado vs Predicción con Error Relativo")
    plt.xlabel("Crímenes Observados")
    plt.ylabel("Crímenes Predichos (Media Posterior)")
    plt.tight_layout()

    n_green = np.sum(colors == "green")
    n_orange = np.sum(colors == "orange")
    n_red = np.sum(colors == "red")

    current_model_info = f"""
    ***{model_name}***
    ----------
    Ancho medio IC : {np.mean(upper - lower):.2f}
    Ancho máximo IC: {np.max(upper - lower):.2f}

    5 áreas con mayor error relativo:
    {legend_text}

    Clasificación por error relativo
      🟢 < 10%      : {n_green}
      🟠 10%-25%    : {n_orange}
      🔴 ≥ 25%      : {n_red}
      Total         : {n_green + n_orange + n_red}
    """
    print(current_model_info)
    models_summary.append(current_model_info)

    plt.show()

print("\n\n\n============ Resumen global para los 5 modelos ============")
for summary in models_summary:
  print(summary)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

observed_data = stan_data["M1"]["y"]

for model_name, fit in fits.items():

    yrep = fit.stan_variable("y_rep")

    pred_mean = yrep.mean(axis=0)

    lower = np.percentile(yrep, 2.5, axis=0)
    upper = np.percentile(yrep, 97.5, axis=0)

    # Error relativo (%)
    rel_error = np.abs(pred_mean - observed_data) / np.maximum(observed_data, 1)

    # Define color categories
    status_color = np.where(
        rel_error < 0.10,
        "Error < 10 %",
        np.where(rel_error < 0.25, "10 % \u2264 Error < 25 %", "Error \u2265 25 %")
    )

    # Create a DataFrame for Plotly Express
    plot_df = pd.DataFrame({
        'Observed': observed_data,
        'Predicted Mean': pred_mean,
        'Error_y_plus': upper - pred_mean,
        'Error_y_minus': pred_mean - lower,
        'Relative Error (%)': rel_error * 100, # Convert to percentage
        'Error Status': status_color,
        'Community Area Number': df_model["Community Area Number"],
        'COMMUNITY AREA NAME': df_model["COMMUNITY AREA NAME"]
    })

    # Sort for consistent color mapping in legend
    plot_df['Error Status'] = pd.Categorical(plot_df['Error Status'],
                                                categories=["Error < 10 %", "10 % \u2264 Error < 25 %", "Error \u2265 25 %"],
                                                ordered=True)


    # Create the scatter plot using Plotly Express
    fig = px.scatter(
        plot_df,
        x='Observed',
        y='Predicted Mean',
        color='Error Status',
        error_y='Error_y_plus',
        error_y_minus='Error_y_minus',
        hover_data={
            'Community Area Number': True,
            'COMMUNITY AREA NAME': True,
            'Observed': ':.0f',
            'Predicted Mean': ':.2f',
            'Relative Error (%)': ':.2f%',
            'Error Status': False, # Hide this from hover
            'Error_y_plus': False,
            'Error_y_minus': False
        },
        title=f"PPC - {model_name}: Observado vs Predicción con Error Relativo",
        labels={
            'Observed': 'Observado',
            'Predicted Mean': 'Predicción',
            'Error Status': 'Rango de Error'
        },
        color_discrete_map={
            "Error < 10 %": "green",
            "10 % \u2264 Error < 25 %": "orange",
            "Error \u2265 25 %": "red"
        }
    )

    # Add identity line
    lim_max = max(plot_df['Observed'].max(), plot_df['Predicted Mean'].max(), upper.max()) * 1.05
    fig.add_trace(go.Scatter(
        x=[0, lim_max],
        y=[0, lim_max],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Recta identidad',
        showlegend=True
    ))

    fig.update_layout(
        hovermode="closest",
        xaxis_title="Crímenes Observados",
        yaxis_title="Crímenes Predichos (Media Posterior)"
    )

    import plotly.io as pio     # la extensión de Colab para VS Code no reconoce correctamente el renderizador de Plotly, por lo que se debe especificar explícitamente.
    pio.renderers.default = "vscode"
    
    fig.show()

# **Fase 5: Selección Formal de Modelos:**

## Introducción a la Fase 5

En esta fase compararemos formalmente los 5 modelos ajustados mediante criterios de información bayesianos que se fundamentan en la distribución posterior completa. Los criterios a utilizar son:

1. **WAIC** (Widely Applicable Information Criterion): Estima el rendimiento predictivo fuera de muestra
2. **PSIS-LOO** (Pareto Smoothed Importance Sampling Leave-One-Out): Aproximación robusta de validación cruzada

Ambos criterios penalizan la complejidad del modelo, evitando overfitting mientras se mantiene capacidad predictiva.


In [ ]:
import arviz as az
print(f"ArviZ: {az.__version__}")


In [ ]:
# ==========================================================
# Fase 5.1: Preparación de objetos InferenceData para ArviZ
# ==========================================================

import warnings
# Las advertencias de WAIC/LOO se mantienen visibles.
import arviz as az
import numpy as np

# Crear objetos InferenceData a partir de los fits de Stan
idata_dict = {}

print("=" * 70)
print("PREPARANDO OBJETOS INFERENCEDATA")
print("=" * 70)

for model_name, fit in fits.items():
    print(f"\n>>> Procesando {model_name}")

    idata = az.from_cmdstanpy(
        posterior=fit,
        log_likelihood="log_lik",
        posterior_predictive="y_rep",
    )

    idata_dict[model_name] = idata

    log_lik_shape = idata.log_likelihood['log_lik'].shape
    print(f"    ✓ Log_lik shape: {log_lik_shape}")
    print(f"    ✓ InferenceData preparado")

print("\n" + "=" * 70)
print("OBJETOS 'INFERENCEDATA' LISTOS")
print("=" * 70)

In [ ]:
# ==========================================================
# Fase 5.2: Cálculo de WAIC (Widely Applicable Information Criterion)
# ==========================================================

waic_results = {}

print("=" * 70)
print("CALCULANDO WAIC")
print("=" * 70)

for model_name, idata in idata_dict.items():
    print(f"\n>>> Calculando WAIC para {model_name}...")

    waic = az.waic(idata, pointwise=True)
    waic_results[model_name] = waic

    print(f"    WAIC (elpd_waic): {waic.elpd_waic:.2f}")
    print(f"    SE: {waic.se:.4f}")
    print(f"    p_waic: {waic.p_waic:.4f}")

print("\n" + "=" * 70)

In [ ]:
# ==========================================================
# Fase 5.3: Cálculo de PSIS-LOO (Pareto Smoothed Importance Sampling LOO)
# ==========================================================

loo_results = {}

print("=" * 70)
print("CALCULANDO PSIS-LOO")
print("=" * 70)

for model_name, idata in idata_dict.items():
    print(f"\n>>> Calculando PSIS-LOO para {model_name}...")

    loo = az.loo(idata, pointwise=True)
    loo_results[model_name] = loo

    print(f"    LOO (elpd_loo): {loo.elpd_loo:.2f}")
    print(f"    SE: {loo.se:.4f}")
    print(f"    p_loo: {loo.p_loo:.4f}")

    # Verificar k-hat (diagnóstico de Pareto)
    k_hat_values = loo.pareto_k
    n_high_k = np.sum(k_hat_values > 0.7)

    if n_high_k > 0:
        print(f"    ⚠️ {n_high_k} observaciones con k-hat > 0.7 (posibles outliers)")
    else:
        print(f"    ✓ Todos los k-hat <= 0.7 (LOO confiable)")

print("\n" + "=" * 70)

In [ ]:
# ==========================================================
# Fase 5.4: Tabla Comparativa Consolidada
# ==========================================================

comparison_data = []

for model_name in ['M1', 'M2', 'M3', 'M4', 'M5']:
    waic = waic_results[model_name]
    loo = loo_results[model_name]

    comparison_data.append({
        'Modelo': model_name,
        'WAIC': round(waic.elpd_waic, 2),
        'SE_WAIC': round(waic.se, 4),
        'elpd_loo': round(loo.elpd_loo, 2),
        'SE_LOO': round(loo.se, 4),
        'Tipo': 'Poisson' if model_name in ['M1', 'M2', 'M3'] else 'Neg. Bin',
        'K_vars': 2 if model_name in ['M1', 'M4'] else 3 if model_name in ['M2', 'M5'] else 3
    })

comparison_df = pd.DataFrame(comparison_data)

# En escala elpd, MAYOR es mejor, así que Δ se mide respecto al máximo
comparison_df['Δ_WAIC'] = comparison_df['WAIC'].max() - comparison_df['WAIC']
comparison_df['Δelpd'] = comparison_df['elpd_loo'].max() - comparison_df['elpd_loo']

print("\n" + "=" * 90)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 90)
print(comparison_df.to_string(index=False))
print("=" * 90)

# Interpretación de Δ_WAIC y Δelpd
print("\n📊 INTERPRETACIÓN (escala elpd: Δ = 0 es el mejor modelo):")
print("  • Δelpd < 4: Modelos con soporte sustancial")
print("  • Δelpd 4-7: Soporte decreciente")
print("  • Δelpd > 10: Modelo descartable")
print("  • Lo mismo aplica para Δ_WAIC")

In [ ]:
# ==========================================================
# Fase 5.4.2 (Complemento): Comparación Formal con az.compare()
# ==========================================================

print("=" * 90)
print("COMPARACIÓN FORMAL DE MODELOS (az.compare)")
print("=" * 90)

compare_df = az.compare(idata_dict, ic="loo", method="stacking")

print("\n" + compare_df.to_string())

print("\n📋 COLUMNAS CLAVE:")
print("  • rank: orden de mejor (0) a peor desempeño predictivo")
print("  • elpd_loo: expected log pointwise predictive density (mayor es mejor)")
print("  • p_loo: número efectivo de parámetros (complejidad penalizada)")
print("  • d_loo: diferencia de elpd_loo respecto al mejor modelo")
print("  • weight: peso de stacking (útil para promediar modelos, 0-1)")
print("  • se: error estándar de elpd_loo")
print("  • dse: error estándar de la diferencia d_loo (¡no es solo la resta de los SE individuales!)")
print("  • warning: True si hay k-hat > 0.7 en ese modelo (revisar Fase 5.6)")

# Visualización estándar de arviz para comparación de modelos
az.plot_compare(compare_df, insample_dev=False)
plt.title("Comparación de Modelos: elpd_loo con Errores Estándar")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Fase 5.5: Visualización de Comparación de Modelos
# ==========================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: WAIC vs Modelo (escala elpd: mayor es mejor)
ax1 = axes[0, 0]
colors_waic = ['green' if x < 4 else 'orange' if x < 7 else 'red' 
               for x in comparison_df['Δ_WAIC']]
ax1.barh(comparison_df['Modelo'], comparison_df['WAIC'], color=colors_waic, alpha=0.7)
ax1.set_xlabel('elpd_waic (mayor es mejor)')
ax1.set_title('WAIC por Modelo')
ax1.grid(True, alpha=0.3, axis='x')

# Gráfico 2: elpd_loo vs Modelo (escala elpd: mayor es mejor)
ax2 = axes[0, 1]
colors_loo = ['green' if x < 4 else 'orange' if x < 7 else 'red' 
              for x in comparison_df['Δelpd']]
ax2.barh(comparison_df['Modelo'], comparison_df['elpd_loo'], color=colors_loo, alpha=0.7)
ax2.set_xlabel('elpd_loo (mayor es mejor)')
ax2.set_title('elpd_loo por Modelo')
ax2.grid(True, alpha=0.3, axis='x')

# Gráfico 3: Diferencias Δ_WAIC
ax3 = axes[1, 0]
bars_waic = ax3.barh(comparison_df['Modelo'], comparison_df['Δ_WAIC'], color=colors_waic, alpha=0.7)
ax3.axvline(x=4, color='orange', linestyle='--', linewidth=2, label='Δ=4 (umbral)')
ax3.axvline(x=7, color='red', linestyle='--', linewidth=2, label='Δ=7 (umbral)')
ax3.set_xlabel('Δ WAIC (diferencia respecto al mejor modelo)')
ax3.set_title('Diferencias Relativas WAIC')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

# Gráfico 4: Diferencias Δelpd (LOO)
ax4 = axes[1, 1]
bars_loo = ax4.barh(comparison_df['Modelo'], comparison_df['Δelpd'], color=colors_loo, alpha=0.7)
ax4.axvline(x=4, color='orange', linestyle='--', linewidth=2, label='Δ=4 (umbral)')
ax4.axvline(x=7, color='red', linestyle='--', linewidth=2, label='Δ=7 (umbral)')
ax4.set_xlabel('Δelpd (diferencia respecto al mejor modelo)')
ax4.set_title('Diferencias Relativas elpd_loo')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Fase 5.6: Diagnóstico Detallado de k-hat (Pareto Shape)
# ==========================================================

print("=" * 80)
print("DIAGNÓSTICO DETALLADO DE K-HAT (OBSERVACIONES INFLUYENTES)")
print("=" * 80)

khat_summary = []

for model_name in ['M1', 'M2', 'M3', 'M4', 'M5']:
    loo = loo_results[model_name]
    k_hat = np.asarray(loo.pareto_k)  # aseguramos array plano de numpy

    # Contadores
    good = int(np.sum(k_hat < 0.5))
    ok = int(np.sum((k_hat >= 0.5) & (k_hat < 0.7)))
    bad = int(np.sum((k_hat >= 0.7) & (k_hat < 1.0)))
    very_bad = int(np.sum(k_hat >= 1.0))

    khat_summary.append({
        'Modelo': model_name,
        'k < 0.5': good,
        'k [0.5-0.7)': ok,
        'k [0.7-1.0)': bad,
        'k >= 1.0': very_bad,
        'LOO_confiable': 'Sí ✓' if (bad + very_bad) == 0 else f'No ⚠️ ({bad + very_bad} malos)'
    })

khat_df = pd.DataFrame(khat_summary)
print("\n" + khat_df.to_string(index=False))

print("\n📋 INTERPRETACIÓN DE K-HAT:")
print("  • k < 0.5: Observación bien-comportada (ideal)")
print("  • 0.5 <= k < 0.7: Observación moderadamente influyente (aceptable)")
print("  • 0.7 <= k < 1.0: Observación muy influyente (LOO puede ser inestable)")
print("  • k >= 1.0: Observación extremadamente influyente (LOO no confiable)")

In [ ]:
# ==========================================================
# Fase 5.7: Recomendación Final y Selección de Modelo Ganador
# ==========================================================

# Identificar modelo ganador por WAIC y elpd_loo (escala elpd: MAYOR es mejor)
best_waic_model = comparison_df.loc[comparison_df['WAIC'].idxmax(), 'Modelo']
best_loo_model = comparison_df.loc[comparison_df['elpd_loo'].idxmax(), 'Modelo']

print("\n" + "=" * 80)
print("RECOMENDACIÓN FINAL DE MODELO")
print("=" * 80)

print(f"\n🏆 MODELO CON MEJOR WAIC: {best_waic_model}")
print(f"🏆 MODELO CON MEJOR elpd_loo: {best_loo_model}")

# Tabla de soporte de modelo
print("\n📊 SOPORTE DE MODELO (por WAIC):")
print("\n" + comparison_df[['Modelo', 'WAIC', 'Δ_WAIC', 'elpd_loo', 'Δelpd']].sort_values('Δ_WAIC').to_string(index=False))

# Clasificación de modelos
print("\n🎯 CLASIFICACIÓN POR EVIDENCIA WAIC:")
models_strong = comparison_df[comparison_df['Δ_WAIC'] < 4]['Modelo'].tolist()
models_moderate = comparison_df[(comparison_df['Δ_WAIC'] >= 4) & (comparison_df['Δ_WAIC'] < 7)]['Modelo'].tolist()
models_weak = comparison_df[comparison_df['Δ_WAIC'] >= 7]['Modelo'].tolist()

print(f"\n  ✅ Soporte sustancial (Δ < 4): {models_strong if models_strong else 'Ninguno'}")
print(f"  ⚠️ Soporte decreciente (4 <= Δ < 7): {models_moderate if models_moderate else 'Ninguno'}")
print(f"  ❌ Soporte débil (Δ >= 7): {models_weak if models_weak else 'Ninguno'}")

# Detalles del ganador
print(f"\n{'=' * 80}")
print(f"PERFIL DETALLADO DEL MODELO RECOMENDADO: {best_waic_model}")
print(f"{'=' * 80}")

ganador_info = comparison_df[comparison_df['Modelo'] == best_waic_model].iloc[0]
print(f"\n  Distribución: {ganador_info['Tipo']}")
print(f"  Número de covariables: {ganador_info['K_vars']}")
print(f"  WAIC (elpd_waic): {ganador_info['WAIC']:.2f} (SE: {ganador_info['SE_WAIC']:.4f})")
print(f"  elpd_loo: {ganador_info['elpd_loo']:.2f} (SE: {ganador_info['SE_LOO']:.4f})")
print(f"  Δ WAIC (respecto mejor): {ganador_info['Δ_WAIC']:.2f}")
print(f"  Δelpd (respecto mejor): {ganador_info['Δelpd']:.2f}")

# Ventajas del modelo ganador
print(f"\n✨ RAZONES PARA SELECCIONAR {best_waic_model}:")
if ganador_info['Δ_WAIC'] < 2:
    print(f"  • Claramente mejor rendimiento predictivo")
    print(f"  • Δ WAIC = {ganador_info['Δ_WAIC']:.2f} indica diferencia sustancial")
elif ganador_info['Δ_WAIC'] < 4:
    print(f"  • Mejor rendimiento predictivo")
    print(f"  • Δ WAIC = {ganador_info['Δ_WAIC']:.2f} es relativamente pequeño")
else:
    print(f"  • Rendimiento comparable a otros modelos")
    print(f"  • Δ WAIC = {ganador_info['Δ_WAIC']:.2f} pero sigue siendo competitivo")

if ganador_info['Tipo'] == 'Neg. Bin':
    print(f"  • Distribución Binomial Negativa: mayor flexibilidad en dispersión")
else:
    print(f"  • Distribución Poisson: más parsimoniosa")

print(f"\n{'=' * 80}")
print(f"✅ FASE 5 COMPLETADA: Modelo {best_waic_model} seleccionado para Fase 6")
print(f"{'=' * 80}")

# Guardar modelo ganador en variable para próximas fases
WINNING_MODEL = best_waic_model
print(f"\n📌 Variable WINNING_MODEL = '{WINNING_MODEL}' lista para uso en próximas fases")

In [ ]:
# ==========================================================
# Fase 5.8: Visualización Avanzada - Comparación LOO con ArviZ
# ==========================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: LOO pointwise comparison
ax1 = axes[0]
loo_compare = []
for model_name in ['M1', 'M2', 'M3', 'M4', 'M5']:
    loo = loo_results[model_name]
    loo_pointwise = loo.loo_i.values
    loo_compare.append(loo_pointwise)

loo_compare = np.array(loo_compare).T  # Transponer a (n_obs, n_models)

# Scatter: cada punto es una observación, cada color es un modelo
for i, model_name in enumerate(['M1', 'M2', 'M3', 'M4', 'M5']):
    ax1.scatter(range(77), loo_compare[:, i], alpha=0.5, label=model_name, s=30)

ax1.set_xlabel('Área Comunitaria')
ax1.set_ylabel('LOO por observación')
ax1.set_title('Contribución a LOO por Área y Modelo')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico 2: Violinplot de distribución LOO por modelo
ax2 = axes[1]
loo_data_list = [loo_compare[:, i] for i in range(5)]
parts = ax2.violinplot(loo_data_list, positions=range(5), widths=0.7, showmeans=True, showmedians=True)

ax2.set_xticks(range(5))
ax2.set_xticklabels(['M1', 'M2', 'M3', 'M4', 'M5'])
ax2.set_ylabel('LOO por observación')
ax2.set_title('Distribución de LOO Pointwise por Modelo')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✅ Gráficos de comparación detallada generados")

## Lectura de WAIC y PSIS-LOO

En estas tablas se usa la escala logarítmica de densidad predictiva esperada:
mayor `elpd_waic` o `elpd_loo` es mejor. En escala de devianza, WAIC = -2 elpd_waic
y menor es mejor. Las diferencias deben interpretarse junto con su incertidumbre,
las advertencias de WAIC y el diagnóstico de Pareto k. Los umbrales de diferencias
fijas dibujados en las gráficas originales son heurísticos, no pruebas de significación.


# **Fase 6: Validación Predictiva Frente a modelos clásicos de Machine Learning**

Hasta este punto hemos comparado varias especificaciones bayesianas con criterios de información (WAIC y PSIS-LOO) y seleccionamos un modelo ganador. Ahora añadimos una referencia predictiva más clásica: modelos de Machine Learning supervisado.

El objetivo de esta fase no es reemplazar el enfoque bayesiano, sino compararlo con alternativas más "simples" (al menos en cuanto a lo que teoría estadística se refiere), pero a la vez, ampliamente usadas en la práctica. Como la variable respuesta es un conteo y el modelo bayesiano incorpora un offset logarítmico de población, la comparación se realizará sobre la escala de conteos esperados.

La idea es responder preguntas como:

- ¿Qué tan bien predice el modelo ganador en datos no usados?
- ¿Qué tan competitivos son modelos clásicos como Ridge, Random Forest o Gradient Boosting?
- ¿Se justifica el esfuerzo del modelo bayesiano cuando se mira solo la capacidad predictiva?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use("seaborn-v0_8")
sns.set_theme(style="whitegrid")

# Modelo ganador elegido en la Fase 5
winning_model_name = WINNING_MODEL
spec = model_specs[winning_model_name]
covariables = spec["covariables"]

print(f"Modelo ganador detectado: {winning_model_name}")
print(f"Distribución: {spec['distribution']}")
print(f"Covariables: {covariables}")

In [ ]:
rng = np.random.default_rng(42)

train_idx, test_idx = train_test_split(
    np.arange(len(df_model)),
    test_size=0.30,
    random_state=42
)

train_df = df_model.iloc[train_idx].copy()
test_df = df_model.iloc[test_idx].copy()

X_train = train_df[covariables].to_numpy()
X_test = test_df[covariables].to_numpy()

y_train = train_df["total_crimes"].astype(int).to_numpy()
y_test = test_df["total_crimes"].astype(int).to_numpy()

offset_train = train_df["offset"].to_numpy()
offset_test = test_df["offset"].to_numpy()

stan_data_train = {
    "N": len(train_df),
    "K": len(covariables),
    "X": X_train,
    "y": y_train,
    "log_offset": offset_train,
}

print(f"Entrenamiento: {len(train_df)} observaciones")
print(f"Prueba: {len(test_df)} observaciones")
print(f"Promedio y_train: {y_train.mean():.2f}")
print(f"Promedio y_test: {y_test.mean():.2f}")

In [ ]:
bayes_model = stan_models[spec["distribution"]]

fit_bayes = bayes_model.sample(
    data=stan_data_train,
    chains=4,
    iter_warmup=400,
    iter_sampling=400,
    parallel_chains=4,
    seed=42,
    show_progress=False
)

alpha_draws = fit_bayes.stan_variable("alpha")
beta_draws = fit_bayes.stan_variable("beta")

if beta_draws.ndim == 1:
    beta_draws = beta_draws[None, :]

if beta_draws.shape[0] != alpha_draws.shape[0]:
    beta_draws = beta_draws.reshape(alpha_draws.shape[0], -1)

log_mu_test = offset_test[:, None] + (X_test @ beta_draws.T) + alpha_draws[None, :]

mu_bayes_draws = np.exp(log_mu_test)
mu_bayes_pred = mu_bayes_draws.mean(axis=1)

mu_bayes_pred = np.clip(mu_bayes_pred, 0.0, None)

print("Predicciones bayesianas calculadas para el conjunto de prueba.")
print(f"Media de predicciones bayesianas en test: {mu_bayes_pred.mean():.2f}")

In [ ]:
feature_names = covariables + ["offset"]

X_train_ml = np.column_stack([X_train, offset_train])
X_test_ml = np.column_stack([X_test, offset_test])

models = {
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingRegressor(
        random_state=42
    )
}

predictions = {}

for name, model in models.items():
    model.fit(X_train_ml, y_train)
    pred = model.predict(X_test_ml)
    pred = np.clip(pred, 0.0, None)
    predictions[name] = pred

predictions["Bayesian (ganador)"] = mu_bayes_pred

def compute_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE_%": np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100,
    }

metrics_rows = []
for name, pred in predictions.items():
    metrics = compute_metrics(y_test, pred)
    metrics_rows.append({"Modelo": name, **metrics})

metrics_df = pd.DataFrame(metrics_rows).sort_values("RMSE").reset_index(drop=True)

display(metrics_df)

metrics_df.to_csv(RESULTS_DIR / "phase6_comparison_metrics.csv", index=False)
print("\nArchivo guardado: phase6_comparison_metrics.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

metrics_long = metrics_df.melt(
    id_vars=["Modelo"],
    value_vars=["RMSE", "MAE"],
    var_name="Métrica",
    value_name="Valor"
)

sns.barplot(
    data=metrics_long,
    x="Valor",
    y="Modelo",
    hue="Métrica",
    ax=axes[0],
    palette=["#4C78A8", "#F58518"]
)
axes[0].set_title("Comparación de error predictivo")
axes[0].set_xlabel("Valor")
axes[0].set_ylabel("Modelo")

for name, pred in predictions.items():
    axes[1].scatter(
        y_test,
        pred,
        alpha=0.7,
        s=55,
        label=name
    )

lim = max(y_test.max(), max(pred.max() for pred in predictions.values())) * 1.05
axes[1].plot([0, lim], [0, lim], "k--", lw=1.5, label="Recta identidad")
axes[1].set_title("Observado vs predicción en prueba")
axes[1].set_xlabel("Observado")
axes[1].set_ylabel("Predicho")
axes[1].legend(loc="best", fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## *Interpretación de los resultados de la Fase 6*

Esta comparación ofrece una referencia predictiva simple y útil, aunque no reemplaza la lógica inferencial del enfoque bayesiano.

En términos prácticos, se espera que:

- el modelo bayesiano gane o compita de manera razonable en métricas de error,
- modelos como Ridge sean una línea base muy sólida,
- Random Forest o Gradient Boosting puedan mejorar ligeramente las predicciones si capturan no-linealidades complejas,
- y el análisis de esta fase ayude a contextualizar si el modelo ganador aporta valor más allá de la mera predicción.

En este notebook, la Fase 6 sirve como una comparación preliminar *(aunque se lleve a cabo en la parte final del trabajo)* y transparente: No se busca “vencer” al modelo bayesiano, sino entender qué tan competitivo es frente a modelos clásicos y qué **trade-off** existe entre **interpretabilidad, incertidumbre y capacidad predictiva**.

# **Fase 7: Síntesis de Resultados y Visualizaciones Finales**

## Síntesis del análisis

M1-M3 usan Poisson; M4-M5 usan Binomial Negativa. Las diferencias entre modelos
son las covariables socioeconómicas listadas en `model_specs`; no se incorporan
efectos categóricos por distrito. Los valores archivados favorecen M5 y muestran
advertencias de PSIS-LOO en los otros modelos. WAIC y PSIS-LOO se reportan como
criterios complementarios; aquí `elpd_waic` y `elpd_loo` son mejores cuando aumentan.
Revise los errores estándar de las diferencias y Pareto k antes de comparar.

La comparación con ML es exploratoria: se usa una partición de 53/24 áreas,
estandarización y selección de especificación previas a la partición. No equivale
a una evaluación anidada independiente. Los resultados archivados se conservan
en `results/archived_results.md`.


In [ ]:
# ==============================================================================
# 7.4: Simulación del Impacto en Política Pública (Código)
# ==============================================================================
# Para demostrar la utilidad práctica del modelo ganador M5, se implementa una 
# simulación contrafáctica: evaluamos el impacto esperado sobre el volumen de 
# criminalidad si se reduce el desempleo en las 5 áreas con mayor vulnerabilidad.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 80)
print("ANÁLISIS CONTRAFÁCTICO Y SIMULACIÓN DE ESCENARIOS (MODELO M5)")
print("=" * 80)

# 1. Identificar las 5 áreas del conjunto de prueba con mayor desempleo estandarizado
top_unemployed_idx = np.argsort(X_test[:, 1])[-5:] 
areas_criticas = test_df.iloc[top_unemployed_idx]['COMMUNITY AREA NAME'].values

# 2. Escenario Base (Predicciones actuales del modelo M5)
pred_base = mu_bayes_pred[top_unemployed_idx]

# 3. Escenario de Intervención (Reducción simulada del desempleo en 1.5 Desviaciones Estándar)
X_test_counterfactual = X_test.copy()
X_test_counterfactual[top_unemployed_idx, 1] -= 1.5  

# 4. Propagación en la posterior de M5
log_mu_counterfactual = offset_test[top_unemployed_idx, None] + \
                        (X_test_counterfactual[top_unemployed_idx, :] @ beta_draws.T) + \
                        alpha_draws[None, :]
pred_counterfactual = np.exp(log_mu_counterfactual).mean(axis=1)

# 5. Tabla de Impacto Comunitario
impacto_df = pd.DataFrame({
    'Área Comunitaria': areas_criticas,
    'Observado Real': y_test[top_unemployed_idx],
    'Predicción Base M5': np.round(pred_base, 1),
    'Predicción Post-Intervención': np.round(pred_counterfactual, 1),
    'Crímenes Mitigados': np.round(pred_base - pred_counterfactual, 1)
})
impacto_df['Mitigación %'] = np.round((impacto_df['Crímenes Mitigados'] / impacto_df['Predicción Base M5']) * 100, 2)

display(impacto_df)

# 6. Gráfico de Escenarios
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(areas_criticas))
ax.bar(x - 0.2, impacto_df['Predicción Base M5'], width=0.4, label='Predicción Base M5 (Actual)', color='#D45B5B', alpha=0.9)
ax.bar(x + 0.2, impacto_df['Predicción Post-Intervención'], width=0.4, label='Predicción con Intervención (Desempleo -1.5 SD)', color='#4A7BB0', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(areas_criticas, rotation=15, ha='right')
ax.set_ylabel('Conteo de Crímenes Esperados')
ax.set_title('Evaluación de Impacto: Reducción Simulada del Desempleo en Zonas Críticas')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n[✓] Fase 7.4: Simulación de escenarios de impacto completada con éxito.")
print("=" * 80)

## Extensiones

La dependencia espacial residual requiere evaluación explícita (mapas, matriz de
vecindad y diagnósticos espaciales). Este notebook no implementa una prueba de
autocorrelación ni un modelo CAR/BYM2. Estas son extensiones pendientes.
Los escenarios al cambiar covariables son predicciones condicionales bajo el modelo;
su interpretación como efectos de políticas requiere supuestos causales adicionales.
